In [2]:
# Import pandas library to work with datasets.
import pandas as pd

In [2]:
# Install openpyxl to enable reading Excels.
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Read the Excels and store them into variables.
gender_df_og = pd.read_excel("../../data_storage/raw_data/voter_data/2020_gender_gen_election.xlsx")
age_df_og = pd.read_excel("../../data_storage/raw_data/voter_data/2020_age_gen_election.xlsx")
race_df_og = pd.read_excel("../../data_storage/raw_data/voter_data/2020_race_gen_election.xlsx")
total_df_og = pd.read_excel("../../data_storage/raw_data/voter_data/2020_total_gen_election.xlsx")

In [ ]:
# Make copies of the datasets and store them into variables.
# We will primarily be working with the following variables in data validation, cleaning, etc.
gender_df = gender_df_og.copy()
age_df = age_df_og.copy()
race_df = race_df_og.copy()
total_df = total_df_og.copy()

In [11]:
# Clean the headers by removing whitespace and linebreaks.
for df in [gender_df, age_df, race_df, total_df]:
    df.columns = df.columns.str.strip().str.replace("\n"," ")

# Rename the columns in the four datasets to avoid any same-name clashes when merging the datasets.
gender_df = gender_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Gender"})
age_df = age_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Age"})
race_df = race_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Race"})
total_df = total_df.rename(columns={"Total Ballots Cast": "Total Ballots Cast - Total"})

In [12]:
# Clean the County values.
for df in [gender_df, age_df, race_df, total_df]:
    df["County"] = df["County"].str.strip()
    df["County"] = df["County"].str.replace("\n"," ")
    df["County"] = df["County"].str.title()

In [13]:
# Join the three demographics datasets.
merged = gender_df.merge(age_df, on="County").merge(race_df, on="County")

In [14]:
# List the columns of the merged dataset to check that they were joined correctly - they were.
merged.columns.tolist()

['County',
 'Total Ballots Cast - Gender',
 'Total Female',
 'Total Male',
 'Total Not Identified',
 'Total Ballots Cast - Age',
 'Age 18-29',
 'Age 30-39',
 'Age 40-49',
 'Age 50-59',
 'Age 60-69',
 'Age 70-79',
 'Age 80-89',
 'Age 90-99',
 'Age 100+',
 'Total Ballots Cast - Race',
 'Total Asian (A)',
 'Total American Indian (AI)',
 'Total Black (B)',
 'Total Federally-Registered (may be of any race) (F)',
 'Total Hispanic (H)',
 'Total Korean (K)',
 'Total Other (O)',
 'Total Not Identified (U)',
 'Total White (W)']

In [ ]:
# If all three values in a row match, the result will be stored as True in the new column "equal". 
# Otherwise, it will be stored as "False".
merged["equal"] = (
    (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Age"]) &
     (merged["Total Ballots Cast - Gender"] == merged["Total Ballots Cast - Race"])
)

# Gathers all of the records shown as "False" and stores them into "mismatches".
mismatches = merged[~merged["equal"]]

In [ ]:
print(mismatches)
# Output: Empty Dataframe.
# This means there are no mismatches in total ballots cast among the three datasets. 

Empty DataFrame
Columns: [County, Total Ballots Cast - Gender, Total Female, Total Male, Total Not Identified, Total Ballots Cast - Age, Age 18-29, Age 30-39, Age 40-49, Age 50-59, Age 60-69, Age 70-79, Age 80-89, Age 90-99, Age 100+, Total Ballots Cast - Race, Total Asian (A), Total American Indian (AI), Total Black (B), Total Federally-Registered (may be of any race) (F), Total Hispanic (H), Total Korean (K), Total Other (O), Total Not Identified (U), Total White (W), equal]
Index: []

[0 rows x 26 columns]


In [17]:
# We can now compare our merged dataset with total_df by joining the two.
compare = merged.merge(total_df, on="County")

# Check it to make sure they were joined correctly.
compare.head()

,County,Total Ballots Cast - Gender,Total Female,Total Male,Total Not Identified,Total Ballots Cast - Age,Age 18-29,Age 30-39,Age 40-49,Age 50-59,...,Total Not Identified (U),Total White (W),equal,Registered Voters,Total Ballots Cast - Total,Straight Party Democrat,Straight Party Republican,Absentee Ballots,Turnout as a Percentage of Registered Voters,Absentee as Percentage of Total Ballots
0,Autauga,27681,15245,12310,126,27681,3701,4051,4619,5512,...,120,21885,True,43482,27813,5055,12916,3494,0.64,0.126
1,Baldwin,110922,60724,50049,149,110922,10689,13025,15928,20071,...,344,101146,True,176642,110214,13483,57259,12903,0.624,0.117
2,Barbour,10551,6059,4483,9,10551,1092,1218,1351,1964,...,24,6110,True,17819,10560,3989,3470,1296,0.593,0.123
3,Bibb,8797,4801,3989,7,8797,1320,1231,1393,1772,...,14,7300,True,14992,9630,1585,5360,577,0.642,0.06
4,Blount,27694,14749,12935,10,27694,3774,3705,4237,5269,...,43,26645,True,41865,27665,1602,17450,1711,0.661,0.062


In [18]:
# Compute the difference between Total Ballots Cast and Total Ballots Cast - Gender (age, and race - remember the last three have the same values)
compare["difference"] = compare["Total Ballots Cast - Total"] - compare["Total Ballots Cast - Gender"]

In [19]:
# Compute the absolute value of the differences just to see how far off they are from each other.
compare["abs_difference"] = compare["difference"].abs()

In [20]:
# Get the proportion of the difference.
compare["difference_percent"] = (compare["abs_difference"]/compare["Registered Voters"])

In [21]:
# Filter to Alabama Black Belt counties.
filtered_compare = compare[compare["County"].isin(["Sumter", "Choctaw", "Greene", "Hale", "Marengo", "Perry", "Dallas", "Wilcox", "Lowndes", "Butler", "Crenshaw", "Montgomery", "Pike", "Bullock", "Macon", "Barbour", "Russell"])]

In [ ]:
filtered_compare.sort_values("difference_percent", ascending=False)[["County", "Total Ballots Cast - Gender", "Total Ballots Cast - Age", "Total Ballots Cast - Race", "Total Ballots Cast - Total", "Registered Voters", "difference_percent", "difference", "abs_difference"]]
# The highest difference_percent is 0.047361	
# So the difference in values between Total Ballots Cast for gender, age, and race aren't too different from the total_df values for Total Ballots Cast

,County,Total Ballots Cast - Gender,Total Ballots Cast - Age,Total Ballots Cast - Race,Total Ballots Cast - Total,Registered Voters,difference_percent,difference,abs_difference
54,Pike,12778,12778,12778,13895,23585,0.047361,1117,1117
50,Montgomery,94347,94347,94347,99326,165228,0.030134,4979,4979
42,Lowndes,6759,6759,6759,6896,10343,0.013246,137,137
5,Bullock,4542,4542,4542,4633,7438,0.012234,91,91
11,Choctaw,7438,7438,7438,7522,11008,0.007631,84,84
52,Perry,5199,5199,5199,5258,8156,0.007234,59,59
32,Hale,7998,7998,7998,7957,12512,0.003277,-41,41
23,Dallas,18053,18053,18053,17949,31771,0.003273,-104,104
6,Butler,9482,9482,9482,9527,14602,0.003082,45,45
20,Crenshaw,6624,6624,6624,6645,10606,0.00198,21,21
